# 23-23 · Тесты аргументов командной строки

Практика к разделу [«Проверяем интерфейс командной строки»](../../site/chapters/glava-23/23-27-testy-cli.html). Повторяет `projects/python/safesort/tests/test_cli.py` в части разбора аргументов — полностью через argparse, без обращения к диску.

## Цель

Написать и запустить тесты для парсера аргументов: все пять подкоманд, значение root по умолчанию для undo и ненулевой код завершения при отсутствии или ошибке в подкоманде.

## Example

In [ ]:
import argparse
from pathlib import Path


def build_parser():
    parser = argparse.ArgumentParser(prog="safesort")
    subparsers = parser.add_subparsers(dest="command", required=True)
    for name in ("scan", "plan", "apply", "duplicates"):
        sub = subparsers.add_parser(name)
        sub.add_argument("root", type=Path)
    undo_parser = subparsers.add_parser("undo")
    undo_parser.add_argument("root", type=Path, nargs="?", default=Path("."))
    return parser


def test_parser_accepts_all_subcommands():
    parser = build_parser()
    for name in ("scan", "plan", "apply", "duplicates"):
        args = parser.parse_args([name, "/tmp/x"])
        assert args.command == name
        assert args.root == Path("/tmp/x")


def test_undo_defaults_to_current_directory():
    parser = build_parser()
    args = parser.parse_args(["undo"])
    assert args.command == "undo"
    assert args.root == Path(".")


def test_missing_subcommand_exits_nonzero():
    import contextlib
    import io

    parser = build_parser()
    buffer = io.StringIO()
    try:
        with contextlib.redirect_stderr(buffer):
            parser.parse_args([])
        podnyalos_iskluchenie = False
        kod = None
    except SystemExit as exc:
        podnyalos_iskluchenie = True
        kod = exc.code

    assert podnyalos_iskluchenie is True
    assert kod != 0


test_parser_accepts_all_subcommands()
test_undo_defaults_to_current_directory()
test_missing_subcommand_exits_nonzero()
print("OK: все три теста прошли.")

## Проверка результата

In [ ]:
parser_dlya_proverki = build_parser()
args_proverki = parser_dlya_proverki.parse_args(["apply", "/home/anna/Downloads"])

assert args_proverki.command == "apply"
assert args_proverki.root == Path("/home/anna/Downloads")
print("Верно: apply разобран с правильным путём.")

## Starter

Заполните отмеченное место. Неизменённый starter не проходит tests.

In [ ]:
def kod_razbora(argv: list[str]) -> int:
    # TODO: call build_parser().parse_args(argv), catch SystemExit.
    raise NotImplementedError


## Task

Напишите `kod_razbora(argv)`: верните 0 для успешного разбора или код SystemExit для help/ошибки.

## Tests

Запустите после task cell: есть основной пример и хотя бы один крайний случай.

In [ ]:
assert kod_razbora(["apply", "/tmp/x"]) == 0
assert kod_razbora(["--help"]) == 0
assert kod_razbora(["zip", "/tmp/x"]) != 0
assert kod_razbora([]) != 0
print("Tests passed")

## Hint

Нормальный `parse_args` просто возвращает Namespace. `--help` и ошибка попадают в `except SystemExit as exc`.

## Solution

<details><summary>Показать решение после собственной попытки</summary>

```python
def kod_razbora(argv: list[str]) -> int:
    try:
        build_parser().parse_args(argv)
    except SystemExit as exc:
        return int(exc.code)
    return 0
```

</details>